Tải dữ liệu từ kaggle

In [ ]:
# Nạp thư viện kagglehub vào Python để sử dụng các hàm quản lý/tải dữ liệu từ Kaggle.
import kagglehub
# Tải phiên bản mới nhất của tập dữ liệu
path = kagglehub.dataset_download("nancyalaswad90/diamonds-prices")
# In ra màn hình dòng chữ thông báo kèm theo giá trị của biến path
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'diamonds-prices' dataset.
Path to dataset files: /kaggle/input/diamonds-prices


Đọc dữ liệu

In [ ]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(path, "Diamonds Prices2022.csv")

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  

Tiền xử lý dữ liệu

In [ ]:
# Nạp công cụ LabelEncoder từ thư viện scikit-learn để mã hóa các biến phân loại dạng chuỗi thành dạng số
from sklearn.preprocessing import LabelEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thuộc tính phân loại bằng LabelEncoder
# Vì LabelEncoder chỉ làm việc trên 1 cột duy nhất tại một thời điểm, ta duyệt qua từng cột cần mã hóa
categorical_cols = ['cut', 'color', 'clarity']

for col in categorical_cols:
    # Khởi tạo một đối tượng LabelEncoder riêng cho từng cột
    le = LabelEncoder()
    # Tiến hành học từ danh sách các giá trị duy nhất và chuyển đổi cột đó sang dạng số (0, 1, 2,...)
    diamonds[col] = le.fit_transform(diamonds[col])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23    2      1        3   61.5   55.0    326  3.95  3.98  2.43
1   0.21    3      1        2   59.8   61.0    326  3.89  3.84  2.31
2   0.23    1      1        4   56.9   65.0    327  4.05  4.07  2.31
3   0.29    3      5        5   62.4   58.0    334  4.20  4.23  2.63
4   0.31    1      6        3   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [ ]:
from sklearn.model_selection import train_test_split

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu train test
X_train, X_test, y_train, y_test = train_test_split(
      X,
      y,
      test_size=0.2,
      random_state=42
    )

# In ra kết quả chia dữ liệu
print(f"Có {X_train.shape[0]} mẫu huấn luyện và {X_test.shape[0]} mẫu kiểm tra")

Có 43020 mẫu huấn luyện và 10755 mẫu kiểm tra


Chuẩn hoá dữ liệu

In [ ]:
# Import công cụ chuẩn hóa MinMaxScaler để đưa các đặc trưng về khoảng [0, 1]
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo đối tượng MinMaxScaler
scaler = MinMaxScaler()
# Học tham số Min-Max trên tập Train và chuyển đổi X_train sang dạng chuẩn hóa [0, 1]
X_train_scaled = scaler.fit_transform(X_train)
# Chuyển đổi X_test dựa trên các tham số Min-Max đã học từ X_train (tránh rò rỉ dữ liệu)
X_test_scaled = scaler.transform(X_test)

Training

In [ ]:
# Import mô hình KNN Regression từ thư viện scikit-learn
from sklearn.neighbors import KNeighborsRegressor
# Import các hàm tính toán độ đo đánh giá hiệu năng mô hình (MAE, MSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import hàm tính căn bậc hai từ thư viện toán học math
from math import sqrt
# Import thư viện đo thời gian thực thi
import time
# Import thư viện pandas để tạo và thao tác với bảng dữ liệu
import pandas as pd
# Import thư viện numpy để tính toán số học
import numpy as np

# 1. KHAI BÁO CÁC GIÁ TRỊ THỬ NGHIỆM
# Tạo danh sách các giá trị K 1 đến 300
k_values = range(1, 301)

# Gán giá trị RMSE tốt nhất ban đầu là vô cùng lớn để dễ dàng cập nhật kỷ lục nhỏ nhất
best_rmse = float('inf')
# Khởi tạo biến rỗng dùng để lưu bộ tham số tối ưu (metric, p, n_neighbors)
best_params = None
# Khởi tạo biến rỗng dùng để lưu đối tượng mô hình tốt nhất tìm được
best_model = None
# Khởi tạo biến rỗng dùng để lưu thời gian thực thi của mô hình tốt nhất
best_times = None

# Khởi tạo danh sách rỗng để chứa kết quả chi tiết của từng giá trị K
k_results = []

# 2. VÒNG LẶP DUYỆT CÁC GIÁ TRỊ K CHO KHOẢNG CÁCH MANHATTAN (p=1)
# In tiêu đề thông báo loại khoảng cách đang thực hiện duyệt K
print("--- Đang chạy duyệt K (1-300) cho khoảng cách Manhattan ---")

# Duyệt qua từng giá trị K trong danh sách k_values từ 1 đến 300
for k in k_values:
    # Khởi tạo mô hình KNN Regression cho khoảng cách Manhattan bằng cách truyền p=1
    model = KNeighborsRegressor(n_neighbors=k, metric='minkowski', p=1)

    # Bắt đầu đo thời gian bằng hàm perf_counter() trước khi huấn luyện và dự đoán
    start_time = time.perf_counter()

    # Huấn luyện mô hình KNN bằng tập dữ liệu Train đã chuẩn hóa bằng MinMaxScaler
    model.fit(X_train_scaled, y_train)
    # Dự đoán giá trị nhãn y trên tập dữ liệu Test đã chuẩn hóa
    y_pred_k = model.predict(X_test_scaled)

    # Kết thúc đo thời gian thực thi bằng hàm perf_counter()
    end_time = time.perf_counter()
    # Tính thời gian thực thi bằng cách lấy thời điểm kết thúc trừ thời điểm bắt đầu
    elapsed_time = end_time - start_time

    # Tính sai số tuyệt đối trung bình (MAE) giữa nhãn thực tế y_test và nhãn dự đoán y_pred_k
    mae = mean_absolute_error(y_test, y_pred_k)
    # Tính sai số bình phương trung bình (MSE) giữa nhãn thực tế y_test và nhãn dự đoán y_pred_k
    mse = mean_squared_error(y_test, y_pred_k)
    # Tính căn bậc hai của MSE để ra độ đo RMSE bằng hàm sqrt()
    rmse = sqrt(mse)
    # Tính hệ số xác định R-squared (R2) giữa nhãn thực tế y_test và nhãn dự đoán y_pred_k
    r2 = r2_score(y_test, y_pred_k)

    # Thêm một từ điển chứa kết quả tính toán của giá trị K hiện tại vào danh sách k_results
    k_results.append({'K': k, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'Time_s': elapsed_time})

    # Kiểm tra nếu chỉ số RMSE của K hiện tại nhỏ hơn kỷ lục RMSE tốt nhất đã lưu
    if rmse < best_rmse:
        # Cập nhật giá trị RMSE kỷ lục mới bằng RMSE của K hiện tại
        best_rmse = rmse
        # Lưu bộ tham số tối ưu tương ứng với giá trị K kỷ lục mới
        best_params = {'metric': 'Manhattan', 'p': 1, 'n_neighbors': k}
        # Lưu lại thời gian thực thi tương ứng của mô hình đạt K kỷ lục mới
        best_times = elapsed_time
        # Lưu lại đối tượng mô hình KNN đạt kết quả RMSE tốt nhất
        best_model = model

# Chuyển đổi danh sách kết quả k_results thành cấu trúc bảng DataFrame của thư viện pandas
df_k = pd.DataFrame(k_results)

# --------------------------------------------------
# TRÍCH XUẤT VÀ IN 10 GIÁ TRỊ K XUNG QUANH K TỐI ƯU
# --------------------------------------------------
# Lấy ra giá trị K tối ưu từ bộ tham số best_params đã được lưu trữ
best_k_val = best_params['n_neighbors']
# Tìm vị trí chỉ số dòng (index) đầu tiên của K tối ưu trong DataFrame df_k
best_idx = df_k[df_k['K'] == best_k_val].index[0]
# Tính tổng số dòng dữ liệu hiện có trong DataFrame df_k
total_rows = len(df_k)

# Tính chỉ số dòng bắt đầu để lấy 5 mẫu phía trước K tối ưu
start_idx = best_idx - 5
# Tính chỉ số dòng kết thúc để lấy 5 mẫu phía sau K tối ưu
end_idx = best_idx + 5

# Kiểm tra nếu chỉ số bắt đầu bé hơn 0 (trường hợp K tối ưu nằm sát đầu danh sách)
if start_idx < 0:
    # Cố định chỉ số bắt đầu bằng 0
    start_idx = 0
    # Ép chỉ số kết thúc bằng 10 để lấy đủ 10 dòng dữ liệu đầu tiên
    end_idx = 10
# Kiểm tra nếu chỉ số kết thúc lớn hơn tổng số dòng (trường hợp K tối ưu nằm sát cuối danh sách)
elif end_idx > total_rows:
    # Cố định chỉ số kết thúc bằng tổng số dòng dữ liệu
    end_idx = total_rows
    # Ép chỉ số bắt đầu lùi về 10 dòng để lấy đủ 10 dòng dữ liệu cuối cùng
    start_idx = total_rows - 10

# Cắt 10 dòng lân cận xung quanh K tối ưu, chọn 6 cột cần thiết và tạo bản sao độc lập bằng .copy()
surrounding_df = df_k.iloc[start_idx:end_idx][['K', 'MAE', 'MSE', 'RMSE', 'R2', 'Time_s']].copy()

# Làm tròn tất cả các giá trị số thực trong các cột MAE, MSE, RMSE, R2, Time_s đến 4 chữ số thập phân
surrounding_df[['MAE', 'MSE', 'RMSE', 'R2', 'Time_s']] = surrounding_df[['MAE', 'MSE', 'RMSE', 'R2', 'Time_s']].round(4)

# In ra tiêu đề thông báo hiển thị bảng 10 giá trị K xung quanh K tối ưu cho khoảng cách Manhattan
print(f"\n=== Bảng 10 giá trị xung quanh K tối ưu (Manhattan) ===")
# In bảng surrounding_df dạng chuỗi văn bản hoàn chỉnh và ẩn cột chỉ số số thứ tự thừa bằng index=False
print(surrounding_df.to_string(index=False))

# In dòng kẻ ngang chứa 50 dấu bằng (=) để phân tách nội dung báo cáo
print("\n" + "="*50)
# In ra từ điển chứa các tham số tối ưu (metric, p, n_neighbors) của khoảng cách Manhattan
print("Tham số tốt nhất (Manhattan):", best_params)
# In giá trị RMSE tốt nhất trên tập Test và làm tròn hiển thị đúng 4 chữ số thập phân (:.4f)
print(f"RMSE tốt nhất trên tập Test (Manhattan): {best_rmse:.4f}")
# In thời gian thực thi tốt nhất trên tập Test và làm tròn hiển thị đúng 4 chữ số thập phân (:.4f)
print(f"Thời gian tốt nhất trên tập Test (Manhattan): {best_times:.4f} giây")
# In dòng kẻ ngang chứa 50 dấu bằng (=) để kết thúc báo cáo
print("="*50)

--- Đang chạy duyệt K (1-300) cho khoảng cách Manhattan ---

=== Bảng 10 giá trị xung quanh K tối ưu (Manhattan) ===
 K      MAE         MSE     RMSE     R2  Time_s
 1 380.1846 593398.2963 770.3235 0.9625  0.3061
 2 352.4554 476864.2263 690.5536 0.9698  0.3551
 3 344.0934 454063.9813 673.8427 0.9713  0.3554
 4 339.8884 432848.6736 657.9124 0.9726  0.3809
 5 340.0095 439100.2047 662.6464 0.9722  0.4062
 6 340.9761 443995.3134 666.3297 0.9719  0.4032
 7 341.3061 452706.7148 672.8348 0.9714  0.4331
 8 343.9899 462134.4087 679.8047 0.9708  0.4403
 9 347.2177 471370.1549 686.5640 0.9702  0.7510
10 350.0809 484690.9078 696.1975 0.9693  0.7638

Tham số tốt nhất (Manhattan): {'metric': 'Manhattan', 'p': 1, 'n_neighbors': 4}
RMSE tốt nhất trên tập Test (Manhattan): 657.9124
Thời gian tốt nhất trên tập Test (Manhattan): 0.3809 giây


Test trên tập train

In [ ]:
# Dự đoán giá kim cương trên tập Train bằng mô hình Manhattan tốt nhất
y_pred_train = best_model.predict(X_train_scaled)

Đánh giá mô hình trên tập train

In [ ]:
# In ra tiêu đề thông báo cho phần đánh giá kết quả trên tập huấn luyện Train
print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TRAIN (MANHATTAN) ===")

# Tính sai số tuyệt đối trung bình (MAE) trên tập Train và in ra kết quả làm tròn đến 4 chữ số thập phân
print("MAE:", round(mean_absolute_error(y_train, y_pred_train), 4))

# Tính sai số bình phương trung bình (MSE) trên tập Train và in ra kết quả làm tròn đến 4 chữ số thập phân
print("MSE:", round(mean_squared_error(y_train, y_pred_train), 4))

# Tính căn bậc hai của MSE (RMSE) trên tập Train và in ra kết quả làm tròn đến 4 chữ số thập phân
print("RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 4))

# Tính hệ số xác định R-squared (R2) trên tập Train và in ra kết quả làm tròn đến 4 chữ số thập phân
print("R^2:", round(r2_score(y_train, y_pred_train), 4))

=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TRAIN (MANHATTAN) ===
MAE: 254.8007
MSE: 251618.4263
RMSE: 501.6158
R^2: 0.9842


Test trên tập test

In [ ]:
# Dự đoán giá kim cương trên tập Test bằng mô hình Manhattan tốt nhất
y_pred_test = best_model.predict(X_test_scaled)


Đánh giá mô hình trên tập test


In [ ]:
# In ra tiêu đề thông báo cho phần đánh giá kết quả trên tập kiểm tra Test
print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (MANHATTAN) ===")

# Tính sai số tuyệt đối trung bình (MAE) trên tập Test và in ra kết quả làm tròn đến 4 chữ số thập phân
print("MAE:", round(mean_absolute_error(y_test, y_pred_test), 4))

# Tính sai số bình phương trung bình (MSE) trên tập Test và in ra kết quả làm tròn đến 4 chữ số thập phân
print("MSE:", round(mean_squared_error(y_test, y_pred_test), 4))

# Tính căn bậc hai của MSE (RMSE) trên tập Test và in ra kết quả làm tròn đến 4 chữ số thập phân
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 4))

# Tính hệ số xác định R-squared (R2) trên tập Test và in ra kết quả làm tròn đến 4 chữ số thập phân
print("R^2:", round(r2_score(y_test, y_pred_test), 4))

=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (MANHATTAN) ===
MAE: 339.8884
MSE: 432848.6736
RMSE: 657.9124
R^2: 0.9726
